# Step 0 — Generate the Egyptian Civil Code QA dataset (no API)

Builds `egyptian_civil_qa.json` with a **free open model running on Colab** (Qwen2.5-7B-Instruct, 4-bit), so no API key or billing is needed.

For every article it writes **3 pairs**: 2 formal-Arabic questions + 1 Egyptian-colloquial question, with **formal-Arabic answers** taken only from the article. Three hand-written examples (articles 147, 163, 374) are shown to the model to set the style.

**File to upload:** `egyptian_civil_code_articles.json`.
**Output:** `egyptian_civil_qa.json` — same format as before (`id`, `question`, `context`, `answer`, `style`), so notebooks 1–3 work unchanged.

**Important**
- Start with `TEST_RUN = True` (about 25 articles) and **read the output**. An open 7B model is weaker than a large API model, especially at Egyptian colloquial, so reviewing a sample matters more here.
- Everything is uploaded from and downloaded to your **laptop** (no Google Drive). The run is long, so run the **"download progress" cell** from time to time. If Colab disconnects, set `RESUME_FROM_UPLOAD = True` in Step 2 and upload that progress file to continue where you stopped.
- Runtime → Change runtime type → **GPU (T4)**.

## Step 1 — Install

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

## Step 2 — Settings and upload

In [ ]:
import json, os, re
from tqdm.auto import tqdm
from google.colab import files

MODEL_ID     = "Qwen/Qwen2.5-7B-Instruct"
TEST_RUN     = True     # True: every 45th article (~25) for a quick check. False: all articles.
RESUME_FROM_UPLOAD = False   # True ONLY after a disconnect: you will be asked to upload your saved progress file
BATCH_SIZE   = 8
MAX_NEW_TOKENS = 900
MIN_CHARS, MAX_ARTICLE_CHARS = 60, 3000
RETRY_FAILED = True

WORK_DIR = "/content"
PROGRESS = f"{WORK_DIR}/qa_progress{'_test' if TEST_RUN else ''}.jsonl"
OUT_PATH = f"{WORK_DIR}/egyptian_civil_qa{'_test' if TEST_RUN else ''}.json"

def upload_as(target_path, prompt):
    """Open the laptop file picker and save the chosen file at target_path (handles 'name (1).json' renames)."""
    print(prompt)
    up = files.upload()
    if not up:
        raise RuntimeError("No file was selected. Run this cell again and choose the file.")
    uploaded_name = list(up.keys())[-1]
    if os.path.abspath(uploaded_name) != os.path.abspath(target_path):
        os.replace(uploaded_name, target_path)

ARTICLES_FILE = "/content/egyptian_civil_code_articles.json"
if not os.path.exists(ARTICLES_FILE):
    upload_as(ARTICLES_FILE, "Select egyptian_civil_code_articles.json from your laptop")
arts = json.load(open(ARTICLES_FILE, encoding="utf-8"))

if RESUME_FROM_UPLOAD and not os.path.exists(PROGRESS):
    upload_as(PROGRESS, f"Select your saved progress file ({os.path.basename(PROGRESS)}) from your laptop")

print("Articles:", len(arts), "| progress file:", PROGRESS)

## Step 3 — Load the model (4-bit)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True),
    device_map="auto",
)
model.eval()
print("Model loaded.")

## Step 4 — Prompt, examples and helpers

In [ ]:
SYSTEM = """أنت تكتب أزواج سؤال وجواب بالعربية من مادة واحدة من القانون المدني المصري.
استخدم نص المادة فقط، ولا تضف أي حكم أو شرط أو استثناء غير وارد فيه.

اكتب دائما 3 أسئلة بزوايا مختلفة (تعريف، حكم، شروط، آثار، استثناءات، مدد، من يلتزم...):
- سؤالان بالفصحى القانونية.
- سؤال واحد بالعامية المصرية البسيطة، بالطريقة التي يسأل بها شخص عادي.

كل سؤال يجب أن يكون مفهوما بذاته دون أن يرى القارئ المادة: لا تكتب "هذه المادة" أو "هذا النص" أو "وفقا للنص" أو رقم المادة داخل السؤال، بل اذكر الموضوع نفسه.

الإجابات كلها بالفصحى، من جملة إلى جملتين، بصياغة وفية لألفاظ المادة، ومستمدة من نصها فقط
(حتى لو كان السؤال بالعامية).

أخرج فقط قائمة JSON بهذا الشكل دون أي نص آخر أو علامات markdown:
[{"question": "...", "answer": "...", "style": "formal"},
 {"question": "...", "answer": "...", "style": "formal"},
 {"question": "...", "answer": "...", "style": "colloquial"}]"""

# Hand-written examples, shown to the model as previous turns (articles 147, 163, 374).
FEWSHOT = [
    ("""المادة 147
(1) العقد شريعة المتعاقدين، فلا يجوز نقضه ولا تعديله إلا باتفاق الطرفين، أو للأسباب التى يقررها القانون.
(2) ومع ذلك إذا طرأت حوادث استثنائية عامة لم يكن فى الوسع توقعها وترتب على حدوثها أن تنفيذ الإلتزام التعاقدى، وان لم يصبح مستحيلا، صار مرهقا للمدين بحيث يهدده بخسارة فادحة، جاز للقاضى تبعا للظروف وسد الموازنة بين مصلحة الطرفين أن يرد الإلتزام المرهق إلى الحد المعقول. ويقع باطلا كل اتفاق على خلاف ذلك.""",
     [{"question": "ما المقصود بأن العقد شريعة المتعاقدين، وما الذي يترتب على ذلك؟",
       "answer": "العقد شريعة المتعاقدين، فلا يجوز نقضه ولا تعديله إلا باتفاق الطرفين أو للأسباب التي يقررها القانون.",
       "style": "formal"},
      {"question": "متى يجوز للقاضي أن يرد الالتزام المرهق إلى الحد المعقول؟",
       "answer": "إذا طرأت حوادث استثنائية عامة لم يكن في الوسع توقعها، وترتب عليها أن صار تنفيذ الالتزام التعاقدي مرهقا للمدين بحيث يهدده بخسارة فادحة وإن لم يصبح مستحيلا، جاز للقاضي تبعا للظروف وبعد الموازنة بين مصلحة الطرفين أن يرد الالتزام المرهق إلى الحد المعقول.",
       "style": "formal"},
      {"question": "هو ينفع الطرفين يتفقوا إن القاضي ميقدرش يخفف الالتزام لو الظروف اتغيرت؟",
       "answer": "لا، فكل اتفاق على خلاف ذلك يقع باطلا.",
       "style": "colloquial"}]),
    ("""المادة 163
كل خطأ سبب ضررا للغير يلزم من ارتكبه بالتعويض.""",
     [{"question": "ما الأثر القانوني المترتب على ارتكاب خطأ يسبب ضررا للغير؟",
       "answer": "يلزم من ارتكب الخطأ بالتعويض.",
       "style": "formal"},
      {"question": "من الذي يلتزم بالتعويض عن الخطأ الذي يسبب ضررا للغير؟",
       "answer": "يلتزم بالتعويض من ارتكب خطأ سبب ضررا للغير.",
       "style": "formal"},
      {"question": "لو حد غلط في حقي وأضرني، يلزمه بإيه؟",
       "answer": "يلزم من ارتكب الخطأ الذي سبب الضرر بالتعويض.",
       "style": "colloquial"}]),
    ("""المادة 374
يتقادم الالتزام بانقضاء خمس عشرة سنة فيما عدا الحالات التى ورد عنها نص خاص فى القانون وفيما عدا الاستثناءات التالية.""",
     [{"question": "ما هي المدة العامة لتقادم الالتزام؟",
       "answer": "يتقادم الالتزام بانقضاء خمس عشرة سنة، فيما عدا الحالات التي ورد عنها نص خاص في القانون وفيما عدا الاستثناءات التالية.",
       "style": "formal"},
      {"question": "هل تسري مدة الخمس عشرة سنة على جميع الالتزامات؟",
       "answer": "لا، فهي لا تسري على الحالات التي ورد عنها نص خاص في القانون ولا على الاستثناءات التالية.",
       "style": "formal"},
      {"question": "الالتزام بيسقط بعد كام سنة في العادة؟",
       "answer": "يتقادم الالتزام عادة بانقضاء خمس عشرة سنة، إلا في الحالات التي ورد عنها نص خاص وفي الاستثناءات التالية.",
       "style": "colloquial"}]),
]

def build_messages(article_text):
    msgs = [{"role": "system", "content": SYSTEM}]
    for art, pairs in FEWSHOT:
        msgs.append({"role": "user", "content": art})
        msgs.append({"role": "assistant", "content": json.dumps(pairs, ensure_ascii=False)})
    msgs.append({"role": "user", "content": article_text})
    return msgs

def parse_pairs(raw):
    """Pull the JSON list out of the model's reply and validate it. Returns [] on failure."""
    m = re.search(r"\[.*\]", raw, flags=re.S)
    if not m:
        return []
    try:
        data = json.loads(m.group(0))
    except Exception:
        return []
    out = []
    for k, p in enumerate(data if isinstance(data, list) else []):
        if not isinstance(p, dict):
            continue
        q, a = str(p.get("question", "")).strip(), str(p.get("answer", "")).strip()
        if not q or not a or q == a:
            continue
        st = str(p.get("style", "")).strip().lower()
        if st not in ("formal", "colloquial"):
            st = "formal" if k < 2 else "colloquial"
        out.append({"question": q, "answer": a, "style": st})
    return out[:3]

@torch.no_grad()
def generate_batch(article_texts, sample=False):
    prompts = [tokenizer.apply_chat_template(build_messages(t), tokenize=False, add_generation_prompt=True)
               for t in article_texts]
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    kw = dict(do_sample=True, temperature=0.5, top_p=0.9) if sample else dict(do_sample=False)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, repetition_penalty=1.05,
                         pad_token_id=tokenizer.pad_token_id, **kw)
    return [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True) for o in out]

import re
_DIAC = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")
def _norm_q(t):
    t = _DIAC.sub("", str(t))
    t = re.sub("[إأآٱ]", "ا", t).replace("ى", "ي").replace("ة", "ه")
    t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.split())

# Questions that cannot be understood without seeing the article ("this article", an article number...).
SELF_REF = re.compile(
    r"(هذه الماده|هذا النص|هذه الفقره|نص الماده|الماده\s*(?:رقم\s*)?\d+"
    r"|(?:وفقا|حسب|بحسب|طبقا)\s+(?:ل)?(?:النص|الماده))")
def is_self_referential(question):
    return bool(SELF_REF.search(_norm_q(question)))


## Step 5 — Generate (resumes automatically)

In [ ]:
def load_progress():
    recs = {}
    if os.path.exists(PROGRESS):
        for line in open(PROGRESS, encoding="utf-8"):
            r = json.loads(line)
            recs[r["i"]] = r          # later lines override earlier ones
    return recs

def run(indices, sample=False, label="generate"):
    for b in tqdm(range(0, len(indices), BATCH_SIZE), desc=label):
        idxs = indices[b:b + BATCH_SIZE]
        raws = generate_batch([arts[i]["context"] for i in idxs], sample=sample)
        with open(PROGRESS, "a", encoding="utf-8") as f:
            for i, raw in zip(idxs, raws):
                pairs = parse_pairs(raw)
                f.write(json.dumps({"i": i, "article": arts[i]["article"], "pairs": pairs,
                                    "raw": "" if pairs else raw}, ensure_ascii=False) + "\n")

candidates = [i for i, a in enumerate(arts) if MIN_CHARS <= len(a["context"]) <= MAX_ARTICLE_CHARS]
if TEST_RUN:
    candidates = candidates[::45]
done = {i for i, r in load_progress().items() if r["pairs"]}
todo = sorted([i for i in candidates if i not in done], key=lambda i: len(arts[i]["context"]))
print(f"{len(candidates)} articles to cover, {len(done)} already done, {len(todo)} to run now")

run(todo)

if RETRY_FAILED:
    failed = [i for i in candidates if not load_progress().get(i, {}).get("pairs")]
    print("Failed first pass:", len(failed))
    if failed:
        run(failed, sample=True, label="retry")

### Step 5b: download your progress (run any time)
Saves the progress file to your laptop. Run it every so often during the long full run, so a Colab disconnect does not lose your work.

In [ ]:
from google.colab import files
if os.path.exists(PROGRESS):
    files.download(PROGRESS)
else:
    print("No progress file yet. Run Step 5 first.")

## Step 6 — Build the dataset and check it

In [ ]:
import pandas as pd

recs = load_progress()
out, n, n_selfref = [], 1, 0
for i in sorted(recs):
    r = recs[i]
    for p in r["pairs"]:
        if is_self_referential(p["question"]):
            n_selfref += 1
            continue
        out.append({"id": n, "question": p["question"], "context": arts[i]["context"],
                    "answer": p["answer"], "style": p["style"]})
        n += 1

json.dump(out, open(OUT_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

covered = len({o["context"] for o in out})
still_failed = [i for i in candidates if not recs.get(i, {}).get("pairs")]
print(f"Pairs: {len(out)} from {covered} articles")
print("Removed (question depends on seeing the article):", n_selfref)
print("Style share:", pd.Series([o["style"] for o in out]).value_counts(normalize=True).round(2).to_dict())
print("Articles with no usable output:", len(still_failed))
print("Saved:", OUT_PATH)

In [ ]:
# READ THESE. Check that answers match the article and that colloquial questions sound Egyptian.
import random
for o in random.Random(1).sample(out, min(12, len(out))):
    print("\n" + "=" * 70)
    print(o["context"][:200].replace("\n", " "), "...")
    print(f"[{o['style']}] Q:", o["question"])
    print("A:", o["answer"])

In [ ]:
from google.colab import files
files.download(OUT_PATH)

**After the test run:** if the examples look good, set `TEST_RUN = False`, run Steps 2–6 again (the model reloads, the progress file is separate) and download the full file. Name it `egyptian_civil_qa.json` before uploading it to notebooks 1 and 2.